Prompt tuning is an important concept, especially when you’re learning PEFT methods (like LoRA, prefix tuning, etc.).

**What is Prompt Tuning? (Simple Definition**)

Prompt tuning is a technique where we do NOT change the model’s weights.
Instead, we learn a small set of trainable prompt vectors that are added to the input.

*Think of it as:*

❌ Not teaching the brain new neurons
✅ Teaching the brain how to listen better





**Normal Prompting vs Prompt Tuning**
1️⃣ **Normal Prompting (No Training)**



Summarize the following text:
<article>


Prompt is written by humans

Model weights are frozen

No learning happens

2️⃣ **Prompt Tuning (Trainable Prompts)**



[ P1 P2 P3 P4 ] + "Summarize the following text: <article>"


P1 P2 P3 ... are virtual tokens

These tokens are learned during training

Model weights stay frozen

💡 You’re training the prompt, not the model.




What Exactly Is Being Trained?

✅ Only prompt embeddings

Component	Trainable?
- Model weights	❌ No
- Prompt embeddings	✅ Yes
- LoRA adapters	❌ No
- Token embeddings	❌ No

Usually:

20–100 virtual tokens

Very small memory footprint




How Prompt Tuning Works Internally

- Add virtual tokens at input layer

- Convert them into embeddings

- These embeddings are learned via backprop

- They steer the attention of the frozen model

- Model outputs improve for the task





[0.12, -0.88, 1.04, ...] + Translate English to French: Hello
You never see these numbers — they are learned vectors.

In [1]:
# Where Prompt Tuning Works Best

# ✅ Classification
# ✅ Simple Q&A
# ✅ Sentiment analysis

In [1]:
!pip install transformers datasets peft accelerate evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00


peft → prompt tuning

evaluate → metrics

accelerate → efficient training

**Step 2: Load Dataset (IMPORTANT)**

In [2]:
from datasets import load_dataset

dataset = load_dataset("squad")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [3]:
dataset["train"][0]


{'id': '5733be284776f41900661182',
 'title': 'University_of_Notre_Dame',
 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}

**Step 3: Prompt Formatting (VERY IMPORTANT)**

In [4]:
def format_qa(example):
    input_text = (
        "Answer the question based on the context.\n\n"
        f"Context: {example['context']}\n\n"
        f"Question: {example['question']}"
    )
    target_text = example["answers"]["text"][0]
    return {"input_text": input_text, "target_text": target_text}


Why this matters:

Prompt tuning learns soft tokens

Hard prompt still matters

Clear instructions = faster convergence

**Step 4: Tokenization**

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

def tokenize(example):
    model_inputs = tokenizer(
        example["input_text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    labels = tokenizer(
        example["target_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

| Parameter          | Reason               |
| ------------------ | -------------------- |
| 512                | T5-small max context |
| 128                | Answers are short    |
| padding=max_length | Stable batching      |
| truncation=True    | Avoid overflow       |


**Step 5: Load Base Model (Frozen)**

In [6]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

At this point:

All weights frozen

No training yet

**Step 6: Apply Prompt Tuning (CORE STEP)**

In [7]:
from peft import PromptTuningConfig, get_peft_model

peft_config = PromptTuningConfig(
    task_type="SEQ_2_SEQ_LM",
    num_virtual_tokens=20,
    tokenizer_name_or_path="google/flan-t5-small"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


trainable params: 20,480 || all params: 76,981,632 || trainable%: 0.0266


Why these parameters?
| Parameter             | Meaning                         | Why                   |
| --------------------- | ------------------------------- | --------------------- |
| num_virtual_tokens=20 | Number of learned prompt tokens | Common starting point |
| task_type             | Seq2Seq                         | T5 is encoder-decoder |
| tokenizer path        | Embedding alignment             | Required              |


**Step 7: Training Setup**



In [8]:
formatted_dataset = dataset.map(format_qa)

tokenized_dataset = formatted_dataset.map(
    tokenize,
    batched=True,
    remove_columns=formatted_dataset["train"].column_names
)


Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./prompt_tuning_qa",
    per_device_train_batch_size=8,
    learning_rate=5e-3,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy="epoch",
    fp16=True,
)


| Method        | LR       |
| ------------- | -------- |
| Full FT       | 2e-5     |
| LoRA          | 2e-4     |
| Prompt tuning | **5e-3** |
Reason:

- Only prompt embeddings

- Need faster updates

- Very small parameter count

**Step 8: Trainer**

In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(2000)),
    eval_dataset=tokenized_dataset["validation"].select(range(500)),
    processing_class=tokenizer,
)


Step 9: Train

In [11]:
trainer.train()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Step,Training Loss
50,0.000000
100,0.000000
150,0.000000
200,0.000000
250,0.000000
300,0.000000
350,0.000000
400,0.000000
450,0.000000
500,0.000000


TrainOutput(global_step=750, training_loss=0.0, metrics={'train_runtime': 171.1839, 'train_samples_per_second': 35.05, 'train_steps_per_second': 4.381, 'total_flos': 1115343028224000.0, 'train_loss': 0.0, 'epoch': 3.0})

What’s happening:

- Loss flows → prompt embeddings

- Model weights untouched

- Virtual tokens adapt attention

**Step 10: Inference Test**

In [18]:
input_text = "Answer the question based on the context.\n\nContext: ...\n\nQuestion: ..."
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))



a b c b c b c d d b d d d d d d d d d d d d d d d
